# Annotation analysis

Sentence-level review analysis for geothermal relevance, frame detection, sentiment, and location extraction.

In [ ]:
import glob
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
ANNOTATION_DIR = ROOT / "annotation"
DB_GLOB = str(ANNOTATION_DIR / "**" / "*.db")
all_db_paths = sorted(glob.glob(DB_GLOB, recursive=True))

def has_required_tables(db_path: str, required=("tasks", "annotations")) -> bool:
    con = sqlite3.connect(db_path)
    try:
        tables = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table'",
            con,
        )["name"].tolist()
    finally:
        con.close()
    return set(required).issubset(tables)

db_paths = [p for p in all_db_paths if has_required_tables(p)]
if not db_paths:
    raise FileNotFoundError(f"No valid annotation databases found under {ANNOTATION_DIR}")

print("Databases:")
for p in db_paths:
    print("-", p)


In [ ]:
def load_annotation_db(db_path: str) -> pd.DataFrame:
    con = sqlite3.connect(db_path)
    try:
        tasks = pd.read_sql_query("SELECT * FROM tasks", con)
        anns = pd.read_sql_query("SELECT * FROM annotations", con)
    finally:
        con.close()

    df = anns.merge(tasks, on="paragraph_uid", how="left", suffixes=("_ann", "_task"))
    return df

df = pd.concat([load_annotation_db(p) for p in db_paths], ignore_index=True)
print(df.shape)
df.head()


In [ ]:
def parse_listish(value):
    if value is None or pd.isna(value):
        return []
    s = str(value).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return []
    return [part.strip() for part in s.split(";") if part.strip()]

def parse_meta(value):
    if value is None or pd.isna(value):
        return {}
    if isinstance(value, dict):
        return value
    try:
        return json.loads(value)
    except Exception:
        return {}

analysis_df = df.copy()
analysis_df["meta"] = analysis_df["meta_json"].map(parse_meta)
analysis_df["sentence_uid"] = analysis_df["paragraph_uid"]
analysis_df["sentence_text"] = analysis_df["meta"].map(lambda m: m.get("sentence_text") or m.get("paragraph_text") or "")
analysis_df["paragraph_text_context"] = analysis_df["meta"].map(lambda m: m.get("paragraph_text") or "")
analysis_df["predicted_frames"] = analysis_df["meta"].map(lambda m: parse_listish(m.get("matched_categories_str") or m.get("aspect_pred")))
analysis_df["predicted_location"] = analysis_df["meta"].map(lambda m: m.get("predicted_location") or m.get("llm_location"))
analysis_df["matched_location"] = analysis_df["meta"].map(lambda m: m.get("matched_location") or m.get("geo_name_matched"))
analysis_df["predicted_sentiment"] = analysis_df["sentiment_pred"]
analysis_df["true_frames"] = analysis_df["matched_categories_true"].map(parse_listish)
analysis_df["geothermal_relevant"] = analysis_df["geothermal_relevant"].map({1: True, 0: False})
analysis_df["sentiment_correct"] = analysis_df["sentiment_correct"].map({1: True, 0: False})
analysis_df["matched_categories_correct"] = analysis_df["matched_categories_correct"].map({1: True, 0: False})
analysis_df["location_correct"] = analysis_df["location_correct"].map({1: True, 0: False})
analysis_df[["annotator", "sentence_uid", "sentence_text", "geothermal_relevant", "predicted_sentiment", "predicted_frames", "predicted_location", "matched_location"]].head()


In [ ]:
summary_metrics = {
    "n_annotations": len(analysis_df),
    "n_unique_sentences": analysis_df["sentence_uid"].nunique(),
    "n_annotators": analysis_df["annotator"].nunique(),
}

for col in ["geothermal_relevant", "sentiment_correct", "matched_categories_correct", "location_correct"]:
    if col in analysis_df.columns and analysis_df[col].notna().any():
        summary_metrics[f"{col}_rate"] = analysis_df[col].mean()

pd.Series(summary_metrics).to_frame("value")


In [ ]:
review_breakdown = pd.DataFrame({
    "geothermal_relevant": analysis_df["geothermal_relevant"].value_counts(dropna=False),
    "sentiment_correct": analysis_df["sentiment_correct"].value_counts(dropna=False),
    "frame_correct": analysis_df["matched_categories_correct"].value_counts(dropna=False),
    "location_correct": analysis_df["location_correct"].value_counts(dropna=False),
}).fillna(0).astype(int)
review_breakdown


In [ ]:
non_geothermal = analysis_df[analysis_df["geothermal_relevant"] == False].copy()
non_geothermal[["sentence_uid", "sentence_text", "predicted_frames", "predicted_sentiment", "predicted_location", "notes"]].head(20)


In [ ]:
overlap = analysis_df.groupby(["sentence_uid", "annotator"]).size().reset_index(name="n")
paired = analysis_df.groupby("sentence_uid").filter(lambda g: g["annotator"].nunique() > 1).copy()
print("Overlap sentences:", paired["sentence_uid"].nunique())

def agreement_rate(frame):
    if frame.empty:
        return None
    return frame.groupby("sentence_uid").apply(lambda g: g.nunique() == 1).mean()

agreement = {
    "geothermal_relevant": agreement_rate(paired[["sentence_uid", "geothermal_relevant"]].dropna()),
    "sentiment_correct": agreement_rate(paired[["sentence_uid", "sentiment_correct"]].dropna()),
    "matched_categories_correct": agreement_rate(paired[["sentence_uid", "matched_categories_correct"]].dropna()),
    "location_correct": agreement_rate(paired[["sentence_uid", "location_correct"]].dropna()),
}
pd.Series(agreement).to_frame("agreement_rate")


In [ ]:
frame_corrections = analysis_df.loc[analysis_df["matched_categories_correct"] == False, [
    "sentence_uid", "sentence_text", "predicted_frames", "true_frames", "keywords_to_add", "notes"
]].copy()
frame_corrections.head(20)


In [ ]:
location_corrections = analysis_df.loc[analysis_df["location_correct"] == False, [
    "sentence_uid", "sentence_text", "predicted_location", "matched_location", "location_true", "notes"
]].copy()
location_corrections.head(20)
